# 02 - Exploratory Data Analysis
## MA Waterways Heatwave Risk Analysis

**Objective**: Explore patterns, distributions, and relationships in the water quality data

**Key Questions**:
- What is the distribution of DO and temperature?
- How do values change over time and seasons?
- What relationships exist between parameters?
- Are there geographic patterns?

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Cleaned Data

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_water_quality.csv')

# Convert date columns back to datetime
date_cols = [col for col in df.columns if 'date' in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Loaded {len(df):,} records with {len(df.columns)} columns")
df.head()

In [ ]:
# Basic information
df.info()

## 2. Descriptive Statistics

Examine central tendencies and dispersion.

In [ ]:
# Statistical summary of key parameters
# Note: Update column names based on actual data
numeric_cols = df.select_dtypes(include=[np.number]).columns[:10]
df[numeric_cols].describe()

In [ ]:
# Temporal coverage
if 'year' in df.columns:
    print("Temporal Coverage:")
    print(f"  Years: {df['year'].min():.0f} - {df['year'].max():.0f}")
    print(f"  Total years: {df['year'].nunique()}")
    print("\nSamples per year:")
    print(df['year'].value_counts().sort_index())

In [ ]:
# Spatial coverage
site_col = [col for col in df.columns if 'site' in col.lower()]
if site_col:
    site_col = site_col[0]
    print(f"Spatial Coverage:")
    print(f"  Unique sites: {df[site_col].nunique():,}")
    print(f"\nTop 10 most sampled sites:")
    print(df[site_col].value_counts().head(10))

## 3. Distribution Analysis

Visualize parameter distributions.

In [ ]:
# Find DO and temperature columns (adjust names as needed)
do_cols = [col for col in df.columns if 'do' in col.lower() or 'oxygen' in col.lower()]
temp_cols = [col for col in df.columns if 'temp' in col.lower()]

print("DO columns:", do_cols)
print("Temperature columns:", temp_cols)

In [ ]:
# Distribution of DO and Temperature
# Update column names based on output above
if do_cols and temp_cols:
    do_col = do_cols[0]
    temp_col = temp_cols[0]
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # DO distribution
    axes[0].hist(df[do_col].dropna(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].axvline(5, color='red', linestyle='--', linewidth=2, label='Critical (5 mg/L)')
    axes[0].set_xlabel('Dissolved Oxygen (mg/L)', fontweight='bold')
    axes[0].set_ylabel('Frequency', fontweight='bold')
    axes[0].set_title('Distribution of Dissolved Oxygen', fontweight='bold', fontsize=13)
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Temperature distribution
    axes[1].hist(df[temp_col].dropna(), bins=50, color='coral', edgecolor='black', alpha=0.7)
    axes[1].axvline(25, color='orange', linestyle='--', linewidth=2, label='Warm (25°C)')
    axes[1].set_xlabel('Water Temperature (°C)', fontweight='bold')
    axes[1].set_ylabel('Frequency', fontweight='bold')
    axes[1].set_title('Distribution of Water Temperature', fontweight='bold', fontsize=13)
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/01_parameter_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()

## 4. Temporal Patterns

Examine how parameters change over time.

In [ ]:
# Monthly patterns
if 'month' in df.columns and do_cols and temp_cols:
    monthly_stats = df.groupby('month')[[do_col, temp_col]].mean()
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    # DO by month
    axes[0].plot(range(1, 13), monthly_stats[do_col], 'o-', linewidth=2, markersize=8, color='steelblue')
    axes[0].axhline(5, color='red', linestyle='--', alpha=0.5)
    axes[0].set_xticks(range(1, 13))
    axes[0].set_xticklabels(month_names)
    axes[0].set_ylabel('Mean DO (mg/L)', fontweight='bold')
    axes[0].set_title('Dissolved Oxygen by Month', fontweight='bold')
    axes[0].grid(alpha=0.3)
    
    # Temperature by month
    axes[1].plot(range(1, 13), monthly_stats[temp_col], 'o-', linewidth=2, markersize=8, color='coral')
    axes[1].axhline(25, color='orange', linestyle='--', alpha=0.5)
    axes[1].set_xticks(range(1, 13))
    axes[1].set_xticklabels(month_names)
    axes[1].set_ylabel('Mean Temperature (°C)', fontweight='bold')
    axes[1].set_title('Water Temperature by Month', fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/02_monthly_patterns.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Yearly trends
if 'year' in df.columns and do_cols and temp_cols:
    yearly_stats = df.groupby('year')[[do_col, temp_col]].mean()
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # DO trend
    axes[0].plot(yearly_stats.index, yearly_stats[do_col], 'o-', linewidth=2, markersize=6, color='steelblue')
    axes[0].set_xlabel('Year', fontweight='bold')
    axes[0].set_ylabel('Mean DO (mg/L)', fontweight='bold')
    axes[0].set_title('Dissolved Oxygen Trend (2005-2020)', fontweight='bold')
    axes[0].grid(alpha=0.3)
    
    # Temperature trend
    axes[1].plot(yearly_stats.index, yearly_stats[temp_col], 'o-', linewidth=2, markersize=6, color='coral')
    axes[1].set_xlabel('Year', fontweight='bold')
    axes[1].set_ylabel('Mean Temperature (°C)', fontweight='bold')
    axes[1].set_title('Water Temperature Trend (2005-2020)', fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/03_yearly_trends.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Relationship Analysis

Examine correlations between parameters.

In [ ]:
# Temperature vs DO scatterplot
if do_cols and temp_cols:
    # Sample for visualization
    df_sample = df[[do_col, temp_col]].dropna()
    if len(df_sample) > 10000:
        df_sample = df_sample.sample(10000, random_state=42)
    
    plt.figure(figsize=(12, 8))
    plt.scatter(df_sample[temp_col], df_sample[do_col], alpha=0.3, s=10, c='steelblue')
    
    # Add regression line
    z = np.polyfit(df_sample[temp_col], df_sample[do_col], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_sample[temp_col].min(), df_sample[temp_col].max(), 100)
    plt.plot(x_line, p(x_line), 'r-', linewidth=3, label=f'Linear fit: y = {z[0]:.3f}x + {z[1]:.2f}')
    
    # Critical lines
    plt.axhline(5, color='darkred', linestyle='--', linewidth=2, alpha=0.7, label='Critical DO')
    plt.axvline(25, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Warm water')
    
    plt.xlabel('Water Temperature (°C)', fontweight='bold', fontsize=12)
    plt.ylabel('Dissolved Oxygen (mg/L)', fontweight='bold', fontsize=12)
    plt.title('Temperature vs Dissolved Oxygen Relationship', fontweight='bold', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/figures/04_temp_do_relationship.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Calculate correlation
    corr = df_sample[temp_col].corr(df_sample[do_col])
    print(f"\nCorrelation between Temperature and DO: {corr:.3f}")

In [ ]:
# Correlation matrix for numeric variables
numeric_cols = df.select_dtypes(include=[np.number]).columns
# Exclude ID and temporal columns
exclude = [col for col in numeric_cols if any(x in col.lower() for x in ['id', 'year', 'month', 'day'])]
analysis_cols = [col for col in numeric_cols if col not in exclude][:10]  # Limit to 10 columns

if len(analysis_cols) > 1:
    corr_matrix = df[analysis_cols].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})
    plt.title('Correlation Matrix of Water Quality Parameters', fontweight='bold', fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig('../outputs/figures/05_correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Key Findings Summary

In [ ]:
# Calculate key statistics
if do_cols and temp_cols:
    print("="*60)
    print("KEY FINDINGS - EXPLORATORY ANALYSIS")
    print("="*60)
    
    print("\n1. DATA COVERAGE:")
    print(f"   Total records: {len(df):,}")
    if 'year' in df.columns:
        print(f"   Time period: {df['year'].min():.0f}-{df['year'].max():.0f}")
    if site_col:
        print(f"   Unique sites: {df[site_col].nunique():,}")
    
    print("\n2. DISSOLVED OXYGEN:")
    print(f"   Mean: {df[do_col].mean():.2f} mg/L")
    print(f"   Median: {df[do_col].median():.2f} mg/L")
    print(f"   Std Dev: {df[do_col].std():.2f} mg/L")
    critical = (df[do_col] < 5).sum()
    print(f"   Critical events (DO < 5): {critical:,} ({critical/len(df)*100:.1f}%)")
    
    print("\n3. TEMPERATURE:")
    print(f"   Mean: {df[temp_col].mean():.2f}°C")
    print(f"   Median: {df[temp_col].median():.2f}°C")
    print(f"   Std Dev: {df[temp_col].std():.2f}°C")
    warm = (df[temp_col] > 25).sum()
    print(f"   Warm events (T > 25): {warm:,} ({warm/len(df)*100:.1f}%)")
    
    print("\n4. COMBINED STRESS:")
    stress = ((df[temp_col] > 25) & (df[do_col] < 5)).sum()
    print(f"   High temp + Low DO: {stress:,} ({stress/len(df)*100:.1f}%)")
    
    print("="*60)

## Next Steps

Proceed to **Notebook 03: Feature Engineering** to:
- Create risk indicators
- Calculate composite risk scores
- Prepare data for modeling